## Example 1: Basic Structure and Unstructure Operations

In [ ]:
import cattrs
from attrs import define
from dataclasses import dataclass
from typing import List, Optional

print("Example 1: Basic Structure and Unstructure Operations")

try:
    # Create converter
    converter = cattrs.Converter()
    print(f"✓ Converter created: {converter}")
    
    # Define data classes
    @define
    class Person:
        name: str
        age: int
        email: Optional[str] = None
    
    @dataclass
    class Address:
        street: str
        city: str
        country: str
    
    print("✓ Data classes defined with attrs and dataclasses")
    
    # Create instances
    person = Person(name="Alice Smith", age=30, email="alice@example.com")
    address = Address(street="123 Main St", city="New York", country="USA")
    
    print(f"✓ Person instance: {person}")
    print(f"✓ Address instance: {address}")
    
    # Unstructure (object -> dict)
    person_dict = converter.unstructure(person)
    address_dict = converter.unstructure(address)
    
    print(f"✓ Person unstructured: {person_dict}")
    print(f"✓ Address unstructured: {address_dict}")
    print(f"  Types: {type(person_dict)}, {type(address_dict)}")
    
    # Structure (dict -> object)
    person_data = {'name': 'Bob Johnson', 'age': 25, 'email': 'bob@example.com'}
    structured_person = converter.structure(person_data, Person)
    
    print(f"✓ Structured person from dict: {structured_person}")
    print(f"  Type: {type(structured_person)}")
    
    print("✓ Basic structure/unstructure operations completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cattrs example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Complex Data Structures and Collections

In [ ]:
import cattrs
from attrs import define
from typing import List, Dict, Union, Optional
from datetime import datetime

print("Example 2: Complex Data Structures and Collections")

try:
    # Complex nested data structures
    @define
    class Product:
        id: int
        name: str
        price: float
        tags: List[str]
    
    @define
    class Customer:
        id: int
        name: str
        email: str
        orders: List[int]
    
    @define
    class Order:
        id: int
        customer_id: int
        products: List[Product]
        total: float
        status: str
    
    converter = cattrs.Converter()
    print("✓ Complex data structures defined")
    
    # Create complex nested structure
    products = [
        Product(1, "Laptop", 999.99, ["electronics", "computers"]),
        Product(2, "Mouse", 29.99, ["electronics", "accessories"])
    ]
    
    order = Order(
        id=100,
        customer_id=1,
        products=products,
        total=1029.98,
        status="pending"
    )
    
    print(f"✓ Complex order created with {len(order.products)} products")
    
    # Unstructure complex object
    order_dict = converter.unstructure(order)
    print(f"✓ Order unstructured to nested dict")
    print(f"  Keys: {list(order_dict.keys())}")
    print(f"  Products count: {len(order_dict['products'])}")
    print(f"  First product: {order_dict['products'][0]['name']}")
    
    # Structure from complex dict
    order_data = {
        "id": 101,
        "customer_id": 2,
        "products": [
            {"id": 3, "name": "Keyboard", "price": 79.99, "tags": ["electronics", "input"]},
            {"id": 4, "name": "Monitor", "price": 299.99, "tags": ["electronics", "display"]}
        ],
        "total": 379.98,
        "status": "completed"
    }
    
    structured_order = converter.structure(order_data, Order)
    print(f"✓ Complex order structured from dict")
    print(f"  Order ID: {structured_order.id}")
    print(f"  Products: {len(structured_order.products)}")
    for p in structured_order.products:
        print(f"    {p.name}: ${p.price}")
    
    # Handle lists of complex objects
    orders_data = [order_dict, order_data]
    orders_list = converter.structure(orders_data, List[Order])
    print(f"✓ List of {len(orders_list)} orders structured successfully")
    
    print("✓ Complex data structure operations completed")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cattrs example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Custom Hooks and Transformations

In [ ]:
import cattrs
from attrs import define
from typing import Dict, Any, Union
from datetime import datetime, date
from enum import Enum

print("Example 3: Custom Hooks and Transformations")

try:
    # Define enum and custom types
    class Status(Enum):
        PENDING = "pending"
        PROCESSING = "processing"  
        COMPLETED = "completed"
        CANCELLED = "cancelled"
    
    @define
    class Task:
        id: int
        title: str
        status: Status
        created_at: datetime
        due_date: date
        metadata: Dict[str, Any]
    
    # Create converter with custom hooks
    converter = cattrs.Converter()
    print("✓ Converter and custom types defined")
    
    # Register custom unstructure hooks
    converter.register_unstructure_hook(
        datetime, 
        lambda dt: dt.isoformat()
    )
    
    converter.register_unstructure_hook(
        date,
        lambda d: d.isoformat()
    )
    
    converter.register_unstructure_hook(
        Status,
        lambda status: status.value
    )
    
    print("✓ Custom unstructure hooks registered")
    
    # Register custom structure hooks  
    converter.register_structure_hook(
        datetime,
        lambda dt_str, _: datetime.fromisoformat(dt_str)
    )
    
    converter.register_structure_hook(
        date,
        lambda d_str, _: date.fromisoformat(d_str)
    )
    
    converter.register_structure_hook(
        Status,
        lambda status_str, _: Status(status_str)
    )
    
    print("✓ Custom structure hooks registered")
    
    # Test with complex object
    task = Task(
        id=1,
        title="Implement feature X",
        status=Status.PROCESSING,
        created_at=datetime(2024, 1, 15, 10, 30),
        due_date=date(2024, 2, 1),
        metadata={"priority": "high", "assignee": "Alice"}
    )
    
    print(f"✓ Task created: {task.title}")
    print(f"  Status: {task.status}")
    print(f"  Created: {task.created_at}")
    
    # Unstructure with custom hooks
    task_dict = converter.unstructure(task)
    print(f"✓ Task unstructured with custom formatting")
    print(f"  Status as string: {task_dict['status']}")
    print(f"  Created as ISO: {task_dict['created_at']}")
    print(f"  Due date as ISO: {task_dict['due_date']}")
    
    # Structure back with custom hooks
    task_data = {
        "id": 2,
        "title": "Review code",
        "status": "completed",
        "created_at": "2024-01-10T09:00:00",
        "due_date": "2024-01-20",
        "metadata": {"priority": "medium", "reviewer": "Bob"}
    }
    
    structured_task = converter.structure(task_data, Task)
    print(f"✓ Task structured from formatted data")
    print(f"  Title: {structured_task.title}")
    print(f"  Status enum: {structured_task.status}")
    print(f"  Created datetime: {structured_task.created_at}")
    print(f"  Due date: {structured_task.due_date}")
    
    print("✓ Custom hooks and transformations completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cattrs example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Union Types and Polymorphism

In [ ]:
import cattrs
from attrs import define
from typing import Union, List, Literal
from abc import ABC, abstractmethod

print("Example 4: Union Types and Polymorphism")

try:
    # Define polymorphic base classes
    @define
    class Shape(ABC):
        type: str
    
    @define
    class Circle(Shape):
        type: Literal["circle"] = "circle"
        radius: float = 0.0
    
    @define 
    class Rectangle(Shape):
        type: Literal["rectangle"] = "rectangle"
        width: float = 0.0
        height: float = 0.0
    
    @define
    class Triangle(Shape):
        type: Literal["triangle"] = "triangle"
        base: float = 0.0
        height: float = 0.0
    
    # Define container with union types
    @define
    class Drawing:
        name: str
        shapes: List[Union[Circle, Rectangle, Triangle]]
    
    converter = cattrs.Converter()
    print("✓ Polymorphic classes defined")
    
    # Configure union handling using discriminator field
    def structure_shape(data, shape_type):
        shape_map = {
            "circle": Circle,
            "rectangle": Rectangle, 
            "triangle": Triangle
        }
        actual_type = shape_map.get(data["type"])
        if actual_type:
            return converter.structure(data, actual_type)
        else:
            raise ValueError(f"Unknown shape type: {data['type']}")
    
    converter.register_structure_hook(
        Union[Circle, Rectangle, Triangle],
        structure_shape
    )
    
    print("✓ Union structure hook registered")
    
    # Create mixed shapes
    shapes = [
        Circle(radius=5.0),
        Rectangle(width=10.0, height=8.0),
        Triangle(base=6.0, height=4.0),
        Circle(radius=3.0)
    ]
    
    drawing = Drawing(name="Mixed Shapes", shapes=shapes)
    print(f"✓ Drawing created with {len(drawing.shapes)} shapes")
    
    # Unstructure preserves type information
    drawing_dict = converter.unstructure(drawing)
    print(f"✓ Drawing unstructured")
    print(f"  Name: {drawing_dict['name']}")
    print(f"  Shapes count: {len(drawing_dict['shapes'])}")
    
    for i, shape_data in enumerate(drawing_dict['shapes']):
        print(f"    Shape {i+1}: {shape_data['type']}")
    
    # Structure back with correct polymorphic types
    drawing_data = {
        "name": "Reconstructed Drawing",
        "shapes": [
            {"type": "circle", "radius": 7.5},
            {"type": "rectangle", "width": 12.0, "height": 6.0},
            {"type": "triangle", "base": 8.0, "height": 5.0}
        ]
    }
    
    structured_drawing = converter.structure(drawing_data, Drawing)
    print(f"✓ Drawing structured with polymorphic shapes")
    print(f"  Name: {structured_drawing.name}")
    print(f"  Shapes: {len(structured_drawing.shapes)}")
    
    for i, shape in enumerate(structured_drawing.shapes):
        print(f"    Shape {i+1}: {type(shape).__name__} - {shape.type}")
    
    # Test union type handling
    shape_union = Union[Circle, Rectangle, Triangle]
    circle_data = {"type": "circle", "radius": 2.5}
    structured_circle = converter.structure(circle_data, shape_union)
    
    print(f"✓ Union type structured: {type(structured_circle).__name__}")
    print(f"  Radius: {structured_circle.radius}")
    
    print("✓ Union types and polymorphism handled successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cattrs example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Advanced Features and Error Handling

In [ ]:
import cattrs
from attrs import define, field, validators
from typing import Optional, Set, Tuple
from cattrs.errors import StructureHandlerNotFoundError, ClassValidationError

print("Example 5: Advanced Features and Error Handling")

try:
    # Define classes with validation. field(validator=...) attaches the validator; `username: str = validators.instance_of(str)`
    # made the validator object the attribute's DEFAULT (measured: {"username": "bob456"} structured with age=<instance_of validator>)
    @define
    class User:
        username: str = field(validator=validators.instance_of(str))
        age: int = field(validator=validators.instance_of(int))
        email: Optional[str] = None
        tags: Set[str] = set()
        coordinates: Tuple[float, float] = (0.0, 0.0)
    
    # Create converter with validation
    converter = cattrs.Converter()
    print("✓ Converter with validated classes created")
    
    # Test successful operations
    user = User(
        username="alice123",
        age=25,
        email="alice@example.com",
        tags={"developer", "python", "open-source"},
        coordinates=(40.7128, -74.0060)  # NYC coordinates
    )
    
    print(f"✓ User created: {user.username}")
    print(f"  Tags: {user.tags}")
    print(f"  Coordinates: {user.coordinates}")
    
    # The validators run on construction
    try:
        User(username=123, age=25)
    except TypeError as e:
        assert e.args[0].startswith("'username' must be <class 'str'>"), e.args
        print(f"✓ Validator rejected username=123: {e.args[0]}")
    else:
        raise AssertionError("User(username=123, age=25) got past the instance_of(str) validator")
    
    # Unstructure with complex types
    user_dict = converter.unstructure(user)
    print(f"✓ User unstructured successfully")
    print(f"  Type of tags: {type(user_dict['tags'])}")
    print(f"  Type of coordinates: {type(user_dict['coordinates'])}")
    assert user_dict == {"username": "alice123", "age": 25, "email": "alice@example.com",
                         "tags": {"developer", "python", "open-source"}, "coordinates": (40.7128, -74.0060)}, user_dict
    
    # Test error handling - invalid data types
    print("✓ Testing error handling...")
    
    try:
        invalid_data = {
            "username": 123,  # Should be string
            "age": "twenty-five",  # Should be int
            "email": "valid@example.com",
            "tags": ["tag1", "tag2"],  # Will be converted to set
            "coordinates": [1.0, 2.0]  # Will be converted to tuple
        }
        
        structured_user = converter.structure(invalid_data, User)
    except ClassValidationError as validation_error:
        # 123 structures to "123" (str() accepts it); "twenty-five" cannot become an int
        attribute_errors, other_errors = validation_error.group_exceptions()
        assert [note for _exc, note in attribute_errors] == ["Structuring class User @ attribute age"], attribute_errors
        assert isinstance(attribute_errors[0][0], ValueError) and not other_errors, (attribute_errors, other_errors)
        print(f"  ✓ Invalid types rejected as expected ({type(validation_error).__name__}, {attribute_errors[0][1]})")
    else:
        raise AssertionError(f"structuring an int field from 'twenty-five' should raise ClassValidationError, got {structured_user!r}")
    
    # Test missing required fields: age has no default, so structuring without it must fail on age
    try:
        incomplete_data = {
            "username": "bob456"
            # Missing required 'age' field
        }
        
        structured_user = converter.structure(incomplete_data, User)
    except ClassValidationError as structure_error:
        attribute_errors, other_errors = structure_error.group_exceptions()
        assert [note for _exc, note in attribute_errors] == ["Structuring class User @ attribute age"], attribute_errors
        assert isinstance(attribute_errors[0][0], KeyError) and not other_errors, (attribute_errors, other_errors)
        print(f"  ✓ Missing field rejected as expected ({type(structure_error).__name__}, missing {attribute_errors[0][0]})")
    else:
        raise AssertionError(f"structuring without the required 'age' should raise ClassValidationError, got {structured_user!r}")
    
    # Test custom error handling
    def safe_structure(data, target_type):
        try:
            return converter.structure(data, target_type)
        except Exception as e:
            print(f"    Safe structure failed: {e}")
            return None
    
    # Test various data formats
    test_cases = [
        {"username": "charlie", "age": 30, "tags": ["python", "data"], "coordinates": (1.0, 2.0)},
        {"username": "diana", "age": 28, "email": "diana@test.com"},
        {"username": "eve", "age": 35, "tags": set(), "coordinates": (0, 0)}
    ]
    
    print("✓ Testing multiple data formats:")
    structured_users = []
    for i, data in enumerate(test_cases):
        result = safe_structure(data, User)
        if result:
            structured_users.append(result)
            print(f"  Case {i+1}: ✓ Success - {result.username}")
        else:
            raise AssertionError(f"Case {i+1}: valid data {data!r} did not structure into User")
    
    print(f"✓ Successfully processed {len(structured_users)} users")
    # lists become sets, (0, 0) becomes floats, omitted fields take their defaults
    assert structured_users == [
        User(username="charlie", age=30, tags={"python", "data"}, coordinates=(1.0, 2.0)),
        User(username="diana", age=28, email="diana@test.com"),
        User(username="eve", age=35, coordinates=(0.0, 0.0)),
    ], structured_users
    assert all(isinstance(c, float) for u in structured_users for c in u.coordinates), structured_users
    
    # Show converter capabilities
    print("✓ Cattrs converter capabilities:")
    print("  - Type-safe structure/unstructure operations")
    print("  - Custom hook registration for complex types")
    print("  - Support for attrs classes and dataclasses")
    print("  - Union type handling with discriminators")
    print("  - Built-in validation support")
    print("  - Comprehensive error handling")
    
    print("✓ Advanced features and error handling demonstrated")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cattrs example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")